In [1]:
import jsonschema
import json
import jsonref
from copy import deepcopy
from benedict import benedict
from perovskite_solar_cell_database.llm_extraction_schema import LLMExtractedPerovskiteSolarCell
from src.nomad_llm_extraction.utils import get_path_b2, update_archive_b2
from glom import glom

/home/pilar/miniconda3/envs/data_ext_tut/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
perov_nomad_jschema=LLMExtractedPerovskiteSolarCell.m_def.m_to_json_schema()

In [3]:
resolved_schema = jsonref.replace_refs(perov_nomad_jschema,jsonschema=True)

In [48]:
archive=json.load(open('temp/10.1002--adfm.202502847-mod.json','r'))
archive=json.load(open('temp/extractions/claude-sonnet-4-20250514/10.1002--aenm.202506634.json','r'))['cells']

In [71]:
KEY_MAPPING = {
    "bandgap": "band_gap",
    "PCE_at_the_start_of_the_experiment": "PCE_at_start",
    "PCE_at_the_end_of_experiment": "PCE_at_end",
    "a_ions": "ions_a_site",
    "b_ions": "ions_b_site",
    "x_ions": "ions_x_site",
    "time": "time", # Keep name, but used for unit check context
    }
REV_KEY_MAPPING={v:k for k,v in KEY_MAPPING.items()}
SPLIT_VALUE_UNIT = ["concentration"]

unit_cond = lambda x,y: 'unit' in x
unit_args = lambda x,y: (y, x['unit'])
SKIP_KEYS = ["additives"] 

def cond_re(section,state):
    return state['name'] in REV_KEY_MAPPING

def re_args(section,state):
    state['a_path']=f'{state['a_p_path']}.{REV_KEY_MAPPING[state['name']]}'
    return state, f'{state['a_p_path']}.{state['name']}'

def rename(jbobj,path,new_path):
    temp_value = deepcopy(jbobj[path])
    jbobj[new_path] = temp_value
    del jbobj[path]
    return jbobj
def remove_uv(jbobj,path,func_args):
    if isinstance(jbobj[path],list):
        jbobj[path]=[i['value'] if (isinstance(i,dict) and 'value' in i) else i for i in jbobj[path] ]
    elif isinstance(jbobj[path],dict) and 'value' in jbobj[path]:
        jbobj[path] = jbobj[path]['value']
    return jbobj
    
def convert_unit(jbobj,path,unit):
    # print(j_path)
    if jbobj[path] is None:
        return jbobj
    items=jbobj[path]
    if isinstance(items,list):
        items = [i['value'] for i in items]
    else:
        items = items['value']
    jbobj[path]=items
    return jbobj

def convert_unit2(jbobj,path,unit):
    # print(j_path)
    if jbobj[path] is None:
        return jbobj
    items=jbobj[path]
    if isinstance(items,list):
        items = [i for i in items]
    else:
        items = items
    jbobj[path]=items
    return jbobj

def cond_del(section,state):
    return state['name'] in SKIP_KEYS

def delete(jbobj,path,func_args):
    del jbobj[path]
    return jbobj

def cond_split_value_unit(section,state):
    return state['name'] in SPLIT_VALUE_UNIT

def split_value_unit(jbobj,path,func_args):
    if jbobj[path] is None:
        return jbobj
    value = jbobj[path]['value']
    unit = jbobj[path]['unit']
    jbobj[path]=value
    jbobj[f'{path}_unit']=unit
    return jbobj

def cond_layer(section,state):
    return state['name'] == 'layers'
def get_layer_order(layers):
    if not layers or not isinstance(layers, list):
        return None
    # Filter layers that have a name and join them
    names = [l["name"] for l in layers if l.get("name")]
    return ",".join(names)
def layer_order()

In [49]:
paths_re=get_path_b2(resolved_schema,'',cond_re,re_args)
paths_unit=get_path_b2(resolved_schema,'',unit_cond,unit_args)
paths=get_path_b2(resolved_schema,'')

In [81]:
paths_re

{'perovskite_composition.bandgap': [{'sname': 'LLMExtractedPerovskiteSolarCell.perovskite_composition.bandgap',
   'name': 'band_gap',
   'p_name': '',
   'p_path': 'properties.perovskite_composition.allOf[1].properties',
   'a_p_path': 'perovskite_composition',
   'path': 'properties.perovskite_composition.allOf[1].band_gap',
   'a_path': 'perovskite_composition.bandgap'},
  'perovskite_composition.band_gap',
  'property'],
 'perovskite_composition.a_ions': [{'sname': 'LLMExtractedPerovskiteSolarCell.perovskite_composition.a_ions',
   'name': 'ions_a_site',
   'p_name': '',
   'p_path': 'properties.perovskite_composition.allOf[1].properties',
   'a_p_path': 'perovskite_composition',
   'path': 'properties.perovskite_composition.allOf[1].ions_a_site',
   'a_path': 'perovskite_composition.a_ions'},
  'perovskite_composition.ions_a_site',
  'property'],
 'perovskite_composition.b_ions': [{'sname': 'LLMExtractedPerovskiteSolarCell.perovskite_composition.b_ions',
   'name': 'ions_b_site',


In [50]:
b_archive=[benedict(deepcopy(i)) for i in archive]

In [72]:
proc_pipeline = {'rename':(cond_re,re_args,rename),\
                 'unit':(unit_cond,unit_args,convert_unit2),\
                 'split_uv':(cond_split_value_unit,None,split_value_unit),\
                 'flatten':(None,None,remove_uv),\
                 'delete':(cond_del,None,delete)}

In [73]:
updated_archive = [benedict(deepcopy(i)) for i in archive]
for i,(proc,(cond,get_func_args,func_apply)) in enumerate(proc_pipeline.items()):
    print(proc)
    paths = get_path_b2(resolved_schema,'',cond,get_func_args)
    updated_archive = update_archive_b2(deepcopy(updated_archive),paths,func_apply)

rename
unit
split_uv
flatten
delete


In [79]:
paths_unit

{'jsc': [{'sname': 'LLMExtractedPerovskiteSolarCell.jsc',
   'name': 'jsc',
   'p_name': '',
   'p_path': 'properties',
   'a_p_path': '',
   'path': 'jsc',
   'a_path': 'jsc'},
  'milliampere / centimeter ** 2',
  'property'],
 'voc': [{'sname': 'LLMExtractedPerovskiteSolarCell.voc',
   'name': 'voc',
   'p_name': '',
   'p_path': 'properties',
   'a_p_path': '',
   'path': 'voc',
   'a_path': 'voc'},
  'volt',
  'property'],
 'active_area': [{'sname': 'LLMExtractedPerovskiteSolarCell.active_area',
   'name': 'active_area',
   'p_name': '',
   'p_path': 'properties',
   'a_p_path': '',
   'path': 'active_area',
   'a_path': 'active_area'},
  'centimeter ** 2',
  'property'],
 'perovskite_composition.band_gap': [{'sname': 'LLMExtractedPerovskiteSolarCell.perovskite_composition.band_gap',
   'name': 'band_gap',
   'p_name': '',
   'p_path': 'properties.perovskite_composition.allOf[1].properties',
   'a_p_path': 'perovskite_composition',
   'path': 'properties.perovskite_composition.allO

In [77]:
up_jobj = update_archive_b2(deepcopy(b_archive),paths_re,rename)

In [78]:
up_jobj2 = update_archive_b2(deepcopy(up_jobj),paths_unit,convert_unit)

In [15]:
up_jobj3 = update_archive_b2(deepcopy(up_jobj2),paths,remove_uv)

In [53]:
glom(archive,'**.ff')

[{'value': 84.02},
 {'value': 84.52},
 {'value': 84.02},
 {'value': 80.89},
 {'value': 82.67}]

In [54]:
glom(updated_archive,'**.ff')

[84.02, 84.52, 84.02, 80.89, 82.67]

In [55]:
glom(archive,'**.PCE_at_the_start_of_the_experiment')

[]

In [56]:
glom(updated_archive,'**.PCE_at_the_start_of_the_experiment')

[]

In [57]:
glom(updated_archive,'**.PCE_at_start')

[]

In [58]:
glom([archive],"**.additives")

[None, None, None, None, None]

In [59]:
glom(updated_archive,"**.additives")

[]

In [75]:
glom(archive,"**.layers")

[[{'name': 'Glass',
   'thickness': None,
   'functionality': 'Substrate',
   'deposition': None,
   'additional_treatment': None},
  {'name': 'FTO',
   'thickness': None,
   'functionality': 'Substrate',
   'deposition': None,
   'additional_treatment': None},
  {'name': '2PACz',
   'thickness': None,
   'functionality': 'Hole-transport',
   'deposition': [{'step_name': None,
     'method': 'Spin-coating',
     'atmosphere': 'Ar',
     'temperature': {'value': 100.0, 'unit': '°C'},
     'duration': {'value': 600.0, 'unit': 's'},
     'antisolvent': None,
     'solution': {'compounds': None,
      'solutes': [{'name': '2PACz',
        'concentration': {'value': 0.001, 'unit': 'mol/L'}}],
      'volume': None,
      'temperature': None,
      'solvents': [{'name': 'IPA', 'volume_fraction': None}]},
     'additional_parameters': {'spin_speed': '3000 rpm',
      'spin_time': '30 s'}}],
   'additional_treatment': None},
  {'name': '(FA0.6MA0.4)Pb(I0.6Br0.4)3',
   'thickness': None,
   'fun

In [74]:
glom(updated_archive,"**.solutes")


[[{'name': '2PACz', 'concentration': 0.001, 'concentration_unit': 'mol/L'}],
 [{'name': 'PbI2', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'PbBr2', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'FAI', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'MABr', 'concentration': 1.35, 'concentration_unit': 'mol/L'}],
 [{'name': '2PACz', 'concentration': 0.001, 'concentration_unit': 'mol/L'}],
 [{'name': 'PbI2', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'PbBr2', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'FAI', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'MABr', 'concentration': 1.35, 'concentration_unit': 'mol/L'}],
 [{'name': '2PACz', 'concentration': 0.001, 'concentration_unit': 'mol/L'}],
 [{'name': 'PbI2', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'PbBr2', 'concentration': 1.35, 'concentration_unit': 'mol/L'},
  {'name': 'FAI', 'concent

In [40]:
from glob import glob
files=glob('temp/*/*/*.json')

In [42]:
for i in files:
    if 'nomad' not in i:
        data=json.load(open(i,'r'))
        if len(glom(data,"**.solutes"))>0:
            print(i)

temp/extractions/claude-sonnet-4-20250514/10.1002--aenm.202506634.json
temp/extractions/claude-sonnet-4-20250514/10.1002--smll.202514110.json
temp/extractions/claude-sonnet-4-20250514/10.1002--smll.202513081.json
temp/extractions/claude-sonnet-4-20250514/10.1002--smll.202512937.json
temp/extractions/proc-extractions/10.1002--adma.202518582.json
temp/extractions/proc-extractions/10.1002--aenm.202506288.json
temp/extractions/proc-extractions/10.1002--adfm.202508510.json
temp/extractions/proc-extractions/10.1002--adma.202512410.json
temp/extractions/proc-extractions/10.1002--adfm.202528728.json
temp/extractions/proc-extractions/10.1002--adma.202520433.json
temp/extractions/proc-extractions/10.1002--adfm.202508262.json
temp/extractions/proc-extractions/10.1002--aenm.202506619.json
temp/extractions/proc-extractions/10.1002--adma.202505694.json
temp/extractions/proc-extractions/10.1002--aenm.202503429.json
temp/extractions/proc-extractions/10.1002--smll.202508334.json
